# Convergence-angle accuracy study — Au [110]

How does diffraction-pattern accuracy degrade as the probe convergence semi-angle increases?

- **Ground truth**: KG ODE with fine z-sampling (128 slices/cell, tight tolerances)
- **Test methods**: Fresnel MS, Angular Spectrum, WPM, KG FWD Lanczos (fixed 8 slices/cell)
- **Input waves**: Aberration-free convergent probes from **5 mrad to 50 mrad** convergence semi-angle
- **Metric**: Relative L2 norm of diffraction pattern vs ground truth

In [1]:
%matplotlib widget

import os
from time import perf_counter
import functools

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".2"

import abtem
import cupy
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from ase.build import bulk, surface

from wide_angle_propagation.propagation_methods import (
    electron_refractive_index,
    energy2wavelength,
    fresnel_propagation_kernel,
    angular_spectrum_propagation_kernel,
    simulate_fresnel_as,
    simulate_wpm,
    simulate_kg_ode_full,
)
from wide_angle_propagation.propagation_methods import _lanczos_expsqrt

abtem.config.set({"device": "gpu"})
abtem.config.set({"precision": "float64"})
jax.config.update("jax_enable_x64", True)

## Build crystal, common parameters, and fixed potentials

In [2]:
# --- Parameters ---
energy = 200e3              # 200 keV
a_Au = 4.076                # lattice parameter (Å)

# --- Build Au [110] orthogonal cell ---
au_bulk = bulk("Au", "fcc", a=a_Au)
au_110 = surface(au_bulk, (1, 1, 0), layers=2)
unit_cell = abtem.orthogonalize_cell(au_110)

z_period = a_Au / np.sqrt(2)       # [110] repeat distance ≈ 2.884 Å
unit_cell.cell[2, 2] = z_period
unit_cell.pbc = [True, True, True]

# Target ~40 Å lateral, ~400 Å thick
nx_rep = int(np.ceil(10 / unit_cell.cell.lengths()[0]))
ny_rep = int(np.ceil(10 / unit_cell.cell.lengths()[1]))
nz_rep = int(np.ceil(100 / z_period))

supercell = unit_cell * (nx_rep, ny_rep, nz_rep)
total_thickness = float(supercell.cell[2, 2])

wavelength = float(energy2wavelength(energy))
print(f"Au [110], {energy/1e3:.0f} keV, λ = {wavelength:.4f} Å")
print(f"Unit cell z-period = {z_period:.4f} Å")
print(f"Supercell: {nx_rep}×{ny_rep}×{nz_rep}, total thickness = {total_thickness:.1f} Å")

# --- Convergence semi-angles to test ---
semi_angle_mrad = np.array([5, 10, 20, 30, 40, 50], dtype=float)
print(f"\nConvergence semi-angles: {semi_angle_mrad} mrad")

# --- Z-sampling configuration ---
GT_SLICES_PER_CELL   = 128   # ground-truth KG ODE
TEST_SLICES_PER_CELL = 8     # fixed sampling for test methods

dz_gt   = z_period / GT_SLICES_PER_CELL
dz_test = z_period / TEST_SLICES_PER_CELL
n_slices_gt   = int(np.round(total_thickness / dz_gt))
n_slices_test = int(np.round(total_thickness / dz_test))

print(f"\nGround truth:  dz = {dz_gt:.4f} Å ({GT_SLICES_PER_CELL} slices/cell, {n_slices_gt} total)")
print(f"Test methods:  dz = {dz_test:.4f} Å ({TEST_SLICES_PER_CELL} slices/cell, {n_slices_test} total)")

Au [110], 200 keV, λ = 0.0251 Å
Unit cell z-period = 2.8836 Å
Supercell: 4×4×35, total thickness = 100.9 Å

Tilt angles: [ 5. 10. 20. 30. 40. 50.] mrad

Ground truth:  dz = 0.0225 Å (128 slices/cell, 4480 total)
Test methods:  dz = 0.3604 Å (8 slices/cell, 280 total)


## Build both potentials once

Since the crystal is the same for all convergence angles, we build each potential once and reuse it.

In [3]:
print("Building ground-truth potential (fine sampling)...")
pot_gt_abtem = abtem.Potential(
    supercell, sampling=0.1, slice_thickness=dz_gt,
    projection="finite", parametrization="lobato",
)
gpts    = tuple(pot_gt_abtem.gpts)
sampling = (float(pot_gt_abtem.sampling[0]), float(pot_gt_abtem.sampling[1]))
pot_gt_arr = jnp.array(cupy.asnumpy(pot_gt_abtem.build(lazy=False).array / dz_gt))
del pot_gt_abtem
cupy.get_default_memory_pool().free_all_blocks()
print(f"  Ground-truth potential shape: {pot_gt_arr.shape}")

print("Building test-method potential (coarser sampling)...")
pot_test_abtem = abtem.Potential(
    supercell, gpts=gpts, slice_thickness=dz_test,
    projection="finite", parametrization="lobato",
)
pot_test_arr = jnp.array(cupy.asnumpy(pot_test_abtem.build(lazy=False).array / dz_test))
del pot_test_abtem
cupy.get_default_memory_pool().free_all_blocks()
print(f"  Test potential shape:         {pot_test_arr.shape}")

# Pre-compute propagation kernels for the test dz
ny_g, nx_g = pot_gt_arr.shape[1], pot_gt_arr.shape[2]
fk = jnp.array(fresnel_propagation_kernel(*gpts, sampling, z=dz_test, energy=energy))
ak = jnp.array(angular_spectrum_propagation_kernel(*gpts, sampling, z=dz_test, energy=energy))

print(f"\nGrid: {gpts}, sampling = ({sampling[0]:.4f}, {sampling[1]:.4f}) Å")

Building ground-truth potential (fine sampling)...
  Ground-truth potential shape: (4480, 116, 116)
Building test-method potential (coarser sampling)...
  Test potential shape:         (280, 116, 116)

Grid: (116, 116), sampling = (0.0994, 0.0994) Å


## KG FWD Lanczos propagator

In [4]:
def simulate_kg_fwd_lanczos(pot_arr, psi_0, dz, E, samp, lanczos_m=130):
    ny, nx = psi_0.shape
    k0_sq = (2 * jnp.pi / energy2wavelength(E)) ** 2
    dy, dx = samp
    fy = jnp.fft.fftfreq(ny, d=dy)
    fx = jnp.fft.fftfreq(nx, d=dx)
    Fx, Fy = jnp.meshgrid(fx, fy)
    kp_sq = (2 * jnp.pi * Fy) ** 2 + (2 * jnp.pi * Fx) ** 2
    n_sq_all = jax.vmap(lambda V: electron_refractive_index(V, E) ** 2)(pot_arr)
    probe_k = jnp.fft.fft2(jnp.asarray(psi_0, dtype=jnp.complex128)) / (ny * nx)
    state = probe_k.ravel()

    @functools.partial(jax.jit, static_argnums=(4, 5, 6))
    def _propagate_one_slice(s, n_sq, k0sq, kpsq, ny_, nx_, m_):
        def matvec(v):
            vg = v.reshape(ny_, nx_)
            conv = jnp.fft.fft2(n_sq * jnp.fft.ifft2(vg))
            return (k0sq * conv - kpsq * vg).ravel()
        return _lanczos_expsqrt(matvec, s, dz, m_)

    for i in range(pot_arr.shape[0]):
        state = _propagate_one_slice(state, n_sq_all[i], k0_sq, kp_sq, ny, nx, lanczos_m)

    exit_k = state.reshape(ny, nx)
    exit_wave = jnp.fft.ifft2(exit_k) * (ny * nx)
    return np.asarray(exit_wave)


def make_convergent_probe(semi_angle_mrad_val, ny, nx, samp_y, samp_x, wavelength_ang):
    """Aberration-free convergent probe with the given convergence semi-angle (mrad).

    Builds a circular aperture in k-space of radius k_max = sin(alpha)/lambda,
    then IFFT2 to get the real-space probe centred on the grid.
    The probe is normalised so that its intensity sums to 1.
    """
    alpha  = semi_angle_mrad_val * 1e-3              # rad
    k_max  = np.sin(alpha) / wavelength_ang          # 1/Å

    fy = np.fft.fftfreq(ny, d=samp_y)
    fx = np.fft.fftfreq(nx, d=samp_x)
    Fx, Fy = np.meshgrid(fx, fy)

    aperture = (np.sqrt(Fx**2 + Fy**2) <= k_max).astype(np.complex128)
    n_pix = float(aperture.real.sum())
    if n_pix > 0:
        aperture /= np.sqrt(n_pix)                  # normalise intensity to 1

    # ifft2 gives probe centred at corner; fftshift moves peak to centre
    probe = np.fft.fftshift(np.fft.ifft2(aperture))
    return jnp.array(probe, dtype=jnp.complex128)


print("Helper functions defined.")

Helper functions defined.


## Sweep convergence semi-angle

For each convergence semi-angle:
1. Build the aberration-free convergent probe
2. Run the fine-sampling KG ODE as ground truth
3. Run each test method at fixed 8-slices/cell sampling
4. Compute Rel L2 of diffraction pattern vs ground truth

In [ ]:
method_names = ["Fresnel MS", "Angular Spectrum", "WPM", "KG FWD Lanczos"]

rel_l2_results = {name: [] for name in method_names}
rms_results    = {name: [] for name in method_names}
timings        = {name: [] for name in method_names}
gt_timings     = []

ny_g, nx_g = pot_gt_arr.shape[1], pot_gt_arr.shape[2]

for alpha in semi_angle_mrad:
    print(f"\n{'='*60}")
    print(f"Convergence semi-angle = {alpha:.0f} mrad")
    print(f"{'='*60}")

    probe = make_convergent_probe(alpha, ny_g, nx_g, sampling[0], sampling[1], wavelength)

    # --- Ground truth ---
    print(f"  KG ODE ground truth (dz={dz_gt:.4f} Å)...", end=" ", flush=True)
    t0 = perf_counter()
    ew_gt, _, _, _ = simulate_kg_ode_full(
        pot_gt_arr, probe, dz_gt, energy, sampling,
        rtol=1e-9, atol=1e-11,
    )
    t_gt = perf_counter() - t0
    dp_gt = np.abs(np.fft.fftshift(np.fft.fft2(np.asarray(ew_gt)))) ** 2
    gt_timings.append(t_gt)
    print(f"{t_gt:.1f}s")

    # --- Test methods ---
    methods_to_run = {
        "Fresnel MS":       lambda p=probe: simulate_fresnel_as(pot_test_arr, p, fk, dz_test, energy),
        "Angular Spectrum": lambda p=probe: simulate_fresnel_as(pot_test_arr, p, ak, dz_test, energy),
        "WPM":              lambda p=probe: simulate_wpm(pot_test_arr, p, dz_test, energy, sampling),
        "KG FWD Lanczos":   lambda p=probe: (simulate_kg_fwd_lanczos(pot_test_arr, p, dz_test, energy, sampling), None, None),
    }

    for name in method_names:
        print(f"  {name}...", end=" ", flush=True)
        t0 = perf_counter()
        result = methods_to_run[name]()
        elapsed = perf_counter() - t0

        ew = np.asarray(result[0])
        dp = np.abs(np.fft.fftshift(np.fft.fft2(ew))) ** 2
        dp_diff = dp - dp_gt
        rel_l2 = float(np.linalg.norm(dp_diff) / np.linalg.norm(dp_gt))
        rms = float(np.sqrt(np.mean(dp_diff**2)))

        rel_l2_results[name].append(rel_l2)
        rms_results[name].append(rms)
        timings[name].append(elapsed)

        print(f"{elapsed:.1f}s, Rel L2 = {rel_l2:.4e}")

print(f"\n{'='*60}")
print("Sweep complete.")


Tilt = 5 mrad
  KG ODE ground truth (dz=0.0225 Å)... 

## Accuracy vs convergence semi-angle

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

markers = {'Fresnel MS': 'o', 'Angular Spectrum': 's', 'WPM': 'D', 'KG FWD Lanczos': '^'}
colors  = {'Fresnel MS': 'C0', 'Angular Spectrum': 'C1', 'WPM': 'C2', 'KG FWD Lanczos': 'C3'}

# --- Left: Rel L2 vs convergence semi-angle ---
for name in method_names:
    ax1.semilogy(semi_angle_mrad, rel_l2_results[name], f'-{markers[name]}',
                 color=colors[name], label=name, markersize=7, linewidth=1.5)

ax1.set_xlabel("Convergence semi-angle (mrad)", fontsize=12)
ax1.set_ylabel("Relative L2 error (diffraction pattern)", fontsize=12)
ax1.set_title("Diffraction-pattern error vs convergence semi-angle\n"
              f"Au [110], {energy/1e3:.0f} keV, {total_thickness:.0f} Å, "
              f"test methods at {TEST_SLICES_PER_CELL} slices/cell", fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# --- Right: runtime vs convergence semi-angle ---
for name in method_names:
    ax2.plot(semi_angle_mrad, timings[name], f'-{markers[name]}',
             color=colors[name], label=name, markersize=7, linewidth=1.5)

ax2.set_xlabel("Convergence semi-angle (mrad)", fontsize=12)
ax2.set_ylabel("Runtime (s)", fontsize=12)
ax2.set_title("Compute cost vs convergence semi-angle", fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Diffraction pattern comparison — selected convergence angles

Row per semi-angle (5, 20, 50 mrad), column per method (+ ground truth).

In [ ]:
selected_angles_mrad = [5.0, 20.0, 50.0]
selected_indices = [int(np.where(semi_angle_mrad == a)[0][0]) for a in selected_angles_mrad]

ny_g, nx_g = pot_gt_arr.shape[1], pot_gt_arr.shape[2]

fig, axes = plt.subplots(
    len(selected_angles_mrad), len(method_names) + 1,
    figsize=(4 * (len(method_names) + 1), 3.5 * len(selected_angles_mrad))
)

for row, (alpha, idx) in enumerate(zip(selected_angles_mrad, selected_indices)):
    probe = make_convergent_probe(alpha, ny_g, nx_g, sampling[0], sampling[1], wavelength)

    # Ground truth
    ew_gt_here, _, _, _ = simulate_kg_ode_full(
        pot_gt_arr, probe, dz_gt, energy, sampling, rtol=1e-9, atol=1e-11,
    )
    dp_gt_here = np.abs(np.fft.fftshift(np.fft.fft2(np.asarray(ew_gt_here)))) ** 2

    col_data = []
    col_labels = []

    # Test methods
    for name in method_names:
        if name == "Fresnel MS":
            result = simulate_fresnel_as(pot_test_arr, probe, fk, dz_test, energy)
        elif name == "Angular Spectrum":
            result = simulate_fresnel_as(pot_test_arr, probe, ak, dz_test, energy)
        elif name == "WPM":
            result = simulate_wpm(pot_test_arr, probe, dz_test, energy, sampling)
        else:
            result = (simulate_kg_fwd_lanczos(pot_test_arr, probe, dz_test, energy, sampling), None, None)

        ew = np.asarray(result[0])
        dp = np.abs(np.fft.fftshift(np.fft.fft2(ew))) ** 2
        col_data.append(dp)
        col_labels.append(name)

    # Determine common colour scale from ground truth
    vmax = np.percentile(dp_gt_here, 99.9)
    vmin = max(vmax * 1e-5, dp_gt_here[dp_gt_here > 0].min())

    # Ground-truth column first
    ax = axes[row, 0]
    ax.imshow(dp_gt_here, norm=LogNorm(vmin=vmin, vmax=vmax), cmap="inferno", origin="lower")
    ax.set_title(f"GT KG ODE\n{alpha:.0f} mrad", fontsize=9)
    ax.axis("off")

    for col, (dp, label) in enumerate(zip(col_data, col_labels)):
        ax = axes[row, col + 1]
        rl2 = rel_l2_results[label][idx]
        ax.imshow(dp, norm=LogNorm(vmin=vmin, vmax=vmax), cmap="inferno", origin="lower")
        ax.set_title(f"{label}\nRel L2={rl2:.2e}", fontsize=9)
        ax.axis("off")

plt.suptitle(
    f"Diffraction patterns — Au [110], {energy/1e3:.0f} keV, {total_thickness:.0f} Å\n"
    f"test methods at {TEST_SLICES_PER_CELL} slices/cell",
    fontsize=11, y=1.01,
)
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
print(f"Au [110], {energy/1e3:.0f} keV, {total_thickness:.0f} Å")
print(f"Ground truth: KG ODE at dz = {dz_gt:.4f} Å ({GT_SLICES_PER_CELL} slices/cell, rtol=1e-9)")
print(f"Test methods: fixed dz = {dz_test:.4f} Å ({TEST_SLICES_PER_CELL} slices/cell)\n")

hdr = f"{'Semi-angle (mrad)':>18s}"
for name in method_names:
    hdr += f" {name:>17s}"
hdr += f" {'GT time (s)':>12s}"
print(hdr)
print("-" * len(hdr))

for i, alpha in enumerate(semi_angle_mrad):
    row = f"{alpha:>18.0f}"
    for name in method_names:
        row += f" {rel_l2_results[name][i]:>17.4e}"
    row += f" {gt_timings[i]:>12.1f}"
    print(row)